This notebook builds the files of the [search development plan](index.qmd) that follow by fixed rules from the files in `data/raw/search-development/`, writes them to `data/processed/search-development/`, and shows the outcomes of the plan's three steps. Step 1 compares the sources @chawinga2019 included with the RDS-Review criteria. Step 2 lists the terms those sources use for each part of the PCC table. Term evidence measures candidate queries in PubMed, Scopus and Web of Science, chooses the query and tests it. Step 3's first version, which chose terms by a yield rule, follows as a record.

`summary_of_outcomes.md` states the outcomes in prose, and `log.md` records the judgments behind the work. The checks come first. If any check finds an error, the notebook stops.

In [ ]:
import math
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
RAW = ROOT / "data" / "raw" / "search-development"
PROCESSED = ROOT / "data" / "processed" / "search-development"
PROCESSED_FILES = {"reference_sources.csv", "counts.csv", "verdict.csv", "statement_matches.csv",
                   "terms.csv", "term_syntax.csv"}
DATABASES = {"PubMed": "pubmed", "Scopus": "scopus", "Web of Science": "web-of-science"}
PARTS = ["sharing", "reuse", "factors that obstruct", "factors that facilitate", "stakeholder perspectives",
         "policies and what drive them", "mechanisms for implementation"]


def path(name):
    return (PROCESSED if name in PROCESSED_FILES else RAW) / name


def read(name):
    return pd.read_csv(path(name), dtype=str, keep_default_na=False)


## Build

Step 1's derived files, `reference_sources.csv`, `counts.csv`, `verdict.csv` and `statement_matches.csv`, from `reference_candidates.csv`, `reference_lookups.csv` and the codes of record. Set, `tier`, Reason, In scope and Eligible apart from year follow the rules in the plan.

Step 2's `terms.csv` and `term_syntax.csv`, from the codes of record, `reference_indexing.csv` and the assignments of record, by the plan's normalisation, syntax and selection rules. Only development set sources supply terms, and a code with a problem from the coding checks supplies none.

In [ ]:
import re

CODES = "reference_codes_qwen3.5-122b.csv"
THEMES = ["factors that obstruct", "factors that facilitate", "stakeholder perspectives",
          "policies and what drive them", "mechanisms for implementation"]
STATEMENTS = [("research data sharing", "researchers"), ("digital curation", "data sharing"),
              ("research data sharing", "open science"), ("research data sharing", "open data")]

cand = read("reference_candidates.csv")
look = read("reference_lookups.csv")
codes = read(CODES)


def write(df, name):
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED / name, index=False, lineterminator="\n")


# one value per source for the single-value fields
single = (codes[codes["field"].isin(["practice", "data_origin", "field", "source_type", "language"])]
          .pivot(index="ref_id", columns="field", values="value"))
themes_of = codes[(codes["field"] == "factors") & (codes["value"] != "none")].groupby("ref_id")["theme"].agg(set)

src = look[["ref_id", "title", "pub_date", "doi", "pmid", "abstract_source"]].merge(
    cand[["ref_id", "label", "cited_in"]], on="ref_id").join(single, on="ref_id")
src["year"] = src["pub_date"].str[:4]
src["set"] = src["year"].astype(int).map(lambda y: "development" if y <= 2014 else "test")
src["tier"] = src["field"].map(lambda f: "1" if f == "health" else "2")
src["coded_on"] = src["ref_id"].map(codes.groupby("ref_id")["coded_on"].first())
src["themes"] = src["ref_id"].map(lambda r: "; ".join(sorted(themes_of.get(r, set()))))


def reasons(s, scope):
    """The codebook.yml exclusion codes a source meets, in the codebook's order. Year and tier count
    only for In scope, not for Eligible apart from year."""
    out = []
    if s["practice"] == "neither":
        out.append("not-data-sharing")
    if not s["themes"]:
        out.append("not-factors")
    if s["data_origin"] == "not-research-data":
        out.append("not-research-data")
    if scope and s["tier"] == "2" and s["source_type"] != "review":
        out.append("tier2-not-review")
    if s["language"] not in ("English", "French"):
        out.append("language")
    if scope and not 2015 <= int(s["year"]) <= 2025:
        out.append("date")
    return out


src["all_reasons"] = src.apply(lambda s: ";".join(reasons(s, True)), axis=1)
src["reason"] = src["all_reasons"].str.split(";").str[0]
src["in_scope"] = src["all_reasons"].map(lambda r: "no" if r else "yes")
src["eligible_apart_from_year"] = src.apply(lambda s: "no" if reasons(s, False) else "yes", axis=1)
src = src[["ref_id", "label", "title", "year", "pub_date", "doi", "pmid", "cited_in", "set", "field", "tier",
           "source_type", "practice", "data_origin", "themes", "language", "coded_on", "abstract_source",
           "eligible_apart_from_year", "in_scope", "reason", "all_reasons"]].sort_values("ref_id")
write(src, "reference_sources.csv")

# counts: sources with each value of each coded field, by set, and codes resting on no passage
setof = src.set_index("ref_id")["set"]
c = codes.assign(set=codes["ref_id"].map(setof), nopassage=codes["passage"].eq("none") & codes["field"].ne("language"))
rows = []
for (field, value, theme), g in c.groupby(["field", "value", "theme"]):
    per = g.groupby("ref_id").agg(set=("set", "first"), nopassage=("nopassage", "all"))
    rows.append(dict(field=field, value=value, theme=theme,
                     development=int((per["set"] == "development").sum()), test=int((per["set"] == "test").sum()),
                     total=len(per), codes_without_passage=int(g["nopassage"].sum())))
for field in ("tier", "eligible_apart_from_year", "in_scope", "reason"):
    for value, g in src.groupby(field):
        rows.append(dict(field=field, value=value or "(none)", theme="",
                         development=int((g["set"] == "development").sum()), test=int((g["set"] == "test").sum()),
                         total=len(g), codes_without_passage=""))
write(pd.DataFrame(rows), "counts.csv")

# verdict: sources eligible apart from year coded for each PCC part, by set
elig = src[src["eligible_apart_from_year"] == "yes"]
has = {"sharing": elig["practice"].isin(["sharing", "both"]), "reuse": elig["practice"].isin(["reuse", "both"])}
has.update({t: elig["themes"].str.contains(t, regex=False) for t in THEMES})
vrows = []
for part, mask in has.items():
    d, t = int((mask & (elig["set"] == "development")).sum()), int((mask & (elig["set"] == "test")).sum())
    verdict = "covered" if d and t else "development only" if d else "test only" if t else "not covered"
    vrows.append(dict(pcc_part=part, development=d, test=t, verdict=verdict))
write(pd.DataFrame(vrows), "verdict.csv")

# Chawinga and Zinn's search statements against titles and abstracts, as phrases and as separate words
squash = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", s.lower()).strip() + " "
text = look.set_index("ref_id").apply(lambda r: squash(r["title"] + " " + r["abstract"]), axis=1)
inscope = set(src.loc[src["in_scope"] == "yes", "ref_id"])
srows = []
for a, b in STATEMENTS:
    for reading in ("phrase", "words"):
        if reading == "phrase":
            hit = lambda t: all(squash(x) in t for x in (a, b))
        else:
            hit = lambda t: all(f" {w} " in t for x in (a, b) for w in squash(x).split())
        ids = sorted(r for r, t in text.items() if hit(t))
        srows.append(dict(statement=f"{a} AND {b}", reading=reading, matched=len(ids),
                          matched_in_scope=sum(r in inscope for r in ids), ref_ids=";".join(ids)))
write(pd.DataFrame(srows), "statement_matches.csv")
print(f"Built reference_sources.csv ({len(src)} sources), counts.csv, verdict.csv and statement_matches.csv.")

# Step 2's terms: recorded terms from the codes, and MeSH headings, author keywords and OpenAlex keywords
# placed under a PCC part by the assignments of record.
ASSIGNMENTS = "term_assignments_qwen3.5-122b.csv"
DBS = ["PubMed", "Scopus", "Web of Science"]
FIELD = {"PubMed": "[tiab]", "Scopus": "TITLE-ABS-KEY", "Web of Science": "TS"}
# Fallback terms from the Candidate terms table on the search strategy page. Reuse always takes its own:
# nearly every development set source coded for reuse is also coded for sharing, so the yield rule can't
# judge a reuse term on sources a sharing term doesn't already reach.
FALLBACK = {"reuse": ["data reuse", "data re-use"]}

dev = src[src["set"] == "development"]
dev_ids = set(dev["ref_id"])
doi_of = src.set_index("ref_id")["doi"]
assign = read(ASSIGNMENTS)
indexing = read("reference_indexing.csv")


def norm(t):
    t = t.lower().strip().strip('"').replace("’", "'")
    return re.sub(r"\s+", " ", re.sub(r"[-‑]", " ", t)).strip()


def stem(w):
    for suffix in ("ies", "es", "s"):
        if w.endswith(suffix) and len(w) > 4:
            return w[:-len(suffix)] + ("y" if suffix == "ies" else "")
    return w


def merge_key(t):
    """Variants that differ only by hyphen, plural or word ending share a key."""
    return " ".join(stem(w) for w in norm(t).split())


def parts_of(value, theme):
    if value in ("none", "neither"):
        return []
    if value == "both":
        return ["sharing", "reuse"]
    if value in ("sharing", "reuse"):
        return [value]
    return [theme] if theme else []


# occurrences: (key, type, part) -> sources carrying it, its variants, and its assignment basis
occ = {}


def add(key, kind, part, ref_id, variant, basis):
    o = occ.setdefault((key, kind, part), {"src": set(), "variants": set(), "basis": basis})
    o["src"].add(ref_id)
    o["variants"].add(variant)


usable = codes[codes["ref_id"].isin(dev_ids) & codes["field"].isin(["practice", "factors"])
               & ~codes["term"].isin(["", "none"]) & (codes["problem"] == "")]
for _, c in usable.iterrows():
    for part in parts_of(c["value"], c["theme"]):
        add(merge_key(c["term"]), "recorded", part, c["ref_id"], norm(c["term"]), "recorded")
value_of = {(a["type"], a["term"]): (a["value"], a["theme"]) for _, a in assign.iterrows()}
for _, i in indexing[indexing["ref_id"].isin(dev_ids)].iterrows():
    value, theme = value_of.get((i["type"], i["term"]), ("none", ""))
    for part in parts_of(value, theme):
        key = i["term"] if i["type"] == "MeSH" else merge_key(i["term"])
        add(key, i["type"], part, i["ref_id"], i["term"] if i["type"] == "MeSH" else norm(i["term"]), value)

# coded sources per PCC part, and fallback terms where a Practice value lacks its own sources
coded = {p: set() for p in PARTS}
for _, c in codes[codes["ref_id"].isin(dev_ids)].iterrows():
    if c["field"] in ("practice", "factors"):
        for part in parts_of(c["value"], c["theme"]):
            coded[part].add(c["ref_id"])
for part, terms_ in FALLBACK.items():
    occ[(terms_[0], "fallback", part)] = {"src": set(), "variants": set(terms_), "basis": "Candidate terms table"}


def written(key, variants, kind, db):
    """One database's written form: MeSH as [mh:noexp] in PubMed only; single words sharing a stem of
    5 or more letters as truncation; otherwise the variants, phrases in quotes, joined with OR."""
    if kind == "MeSH":
        return f'"{key}"[mh:noexp]' if db == "PubMed" else None
    vs = sorted(variants)
    if all(" " not in v for v in vs) and (len(vs) > 1 or stem(vs[0]) != vs[0]):
        root = min((stem(v) for v in vs), key=len)
        while not all(v.startswith(root) for v in vs):
            root = root[:-1]
        forms = [root + "*"] if len(root) >= 5 else vs
    else:
        forms = [f'"{v}"' if " " in v else v for v in vs]
    if db == "PubMed":
        return " OR ".join(f"{f}[tiab]" for f in forms)
    q = " OR ".join(forms)
    return f"TITLE-ABS-KEY({q})" if db == "Scopus" else f"TS=({q})"


# A sharing or reuse term goes to step 3 only if it names data, or is a MeSH heading the model placed by
# its scope note: the Concept is the sharing and reuse of research data, and a term such as "use" or
# "sharing" alone searches for any use or sharing.
names_data = lambda term: re.search(r"data", term.lower()) is not None
shown = lambda key, o, kind: key if kind == "MeSH" else min(o["variants"], key=lambda v: (len(v), v))
syntax_rows, term_rows = [], []
for db in DBS:
    here = {k: o for k, o in occ.items() if not (k[1] == "MeSH" and db != "PubMed")}
    seen = set()
    for (key, kind, part), o in sorted(here.items()):
        if (key, kind) not in seen:
            seen.add((key, kind))
            syntax_rows.append(dict(term=shown(key, o, kind), type=kind, database=db,
                                    variants="; ".join(sorted(o["variants"])),
                                    written_form=written(key, o["variants"], kind, db)))
    for part in PARTS:
        ranked = sorted(((k, o) for k, o in here.items() if k[2] == part),
                        key=lambda x: (-len(x[1]["src"]), x[0][0], x[0][1]))
        matched = set()
        for (key, kind, _), o in ranked:
            if part in ("sharing", "reuse") and kind != "MeSH" and not names_data(shown(key, o, kind)):
                continue
            row = dict(term=shown(key, o, kind), type=kind, database=db,
                       field="[mh:noexp]" if kind == "MeSH" else FIELD[db], pcc_part=part,
                       sources_in_part=len(coded[part]), assignment_basis=o["basis"], count=len(o["src"]))
            if part in ("sharing", "reuse"):
                row.update(sources_added="", dois=";".join(doi_of[r] or r for r in sorted(o["src"])), status="")
            else:
                added = o["src"] - matched
                if not added:
                    continue
                matched |= added
                row.update(sources_added=len(added), dois=";".join(doi_of[r] or r for r in sorted(added)),
                           status="to screening")
            term_rows.append(row)
write(pd.DataFrame(syntax_rows), "term_syntax.csv")
write(pd.DataFrame(term_rows), "terms.csv")
print(f"Built term_syntax.csv ({len(syntax_rows)} written forms) and terms.csv ({len(term_rows)} rows).")

# precision_screening.html: the three precision samples in one page for screening, from the template beside
# this notebook, with the exclusion codes from codebook.yml
import json
import yaml

records, cols = [], None
for db, slug in DATABASES.items():
    p = path(f"precision_sample_{slug}.csv")
    if not p.exists():
        continue
    s = read(p.name)
    cols = cols or list(s.columns) + ([] if "note" in s.columns else ["note"])
    for i, r in enumerate(s.to_dict("records")):
        records.append(dict(db=db, slug=slug, n=i + 1, of=len(s), id=r["record_id"], doi=r["doi"], title=r["title"],
                            year=r["year"], source=r["abstract_source"], abstract=r["abstract"],
                            decision=r.get("decision", ""), code=r.get("exclusion_code", ""), note=r.get("note", "")))
if records:
    codes_yml = yaml.safe_load((ROOT / "codebook.yml").read_text(encoding="utf-8"))["exclusion_reasons"]
    data = json.dumps(dict(records=records, codes=codes_yml, cols=cols), ensure_ascii=False).replace("</", "<\\/")
    present = [db for db in DATABASES if any(r["db"] == db for r in records)]
    template = (ROOT / "analysis" / "00_search-development" / "precision_screening_template.html").read_text(encoding="utf-8")
    page = template.replace("__DATA__", data).replace("__DBS__", ", ".join(present))
    (PROCESSED / "precision_screening.html").write_text(page, encoding="utf-8")
    print(f"Built precision_screening.html ({len(records)} records from {', '.join(present)}).")


In [ ]:
steps_1_2 = ["reference_candidates.csv", "reference_sources.csv", CODES, "counts.csv",
             "verdict.csv", "comparison.csv", "statement_matches.csv", "terms.csv", "term_syntax.csv"]
missing = [n for n in steps_1_2 if not path(n).exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))
else:
    cand, src, codes, counts, verdict, comparison, statements, terms, syntax = (read(n) for n in steps_1_2)
have_tests = path("query_tests.csv").exists()

## Checks

In [ ]:
if have_data:
    errors = []
    ids = set(src["ref_id"])
    if set(cand["ref_id"]) != ids:
        errors.append(f"reference_sources.csv and reference_candidates.csv list different sources: {sorted(set(cand['ref_id']) ^ ids)}.")
    unknown = set(codes["ref_id"]) - ids
    if unknown:
        errors.append(f"{CODES} has sources not in reference_sources.csv: {sorted(unknown)}.")
    for f in ("practice", "data_origin", "field", "source_type", "language"):
        n = codes[codes["field"] == f].groupby("ref_id").size().reindex(sorted(ids), fill_value=0)
        if (n != 1).any():
            errors.append(f"{int((n != 1).sum())} source(s) lack exactly one {f} code: {sorted(n[n != 1].index)}.")
    pairs = set(zip(syntax["term"], syntax["type"], syntax["database"]))
    unwritten = [t for t in zip(terms["term"], terms["type"], terms["database"]) if t not in pairs]
    if unwritten:
        errors.append(f"{len(unwritten)} row(s) in terms.csv have no written form in term_syntax.csv: {unwritten[:5]}.")
    if errors:
        raise ValueError("\n".join(errors))
    print("All checks passed.")
    flagged = codes[codes["problem"] != ""]
    if len(flagged):
        print(f"{len(flagged)} code(s) in {CODES} carry a problem from the coding checks:")
        display(flagged[["ref_id", "field", "value", "problem", "passage"]].reset_index(drop=True))


## Step 1. Comparability assessment

The reference set holds the references @chawinga2019 name in Table 2, cite in sections 4 and 5, or cite for a country section 4.3 lists. A local model coded each one from its title and abstract. Sources first published from 2007 to 2014 form the development set, which step 2 draws terms from. Sources first published from 2015 to July 2018 form the test set, which step 3 measures coverage against.

In [ ]:
if have_data:
    sets = pd.crosstab(src["set"], src["eligible_apart_from_year"], margins=True, margins_name="total")
    sets.columns.name = "eligible apart from year"
    display(sets)
    reasons = (src.assign(reason=src["reason"].replace("", "(none)"))
               .groupby(["reason", "set"]).size().unstack(fill_value=0))
    display(reasons)

### Codes

Each count is the number of sources with that value. `codes_without_passage` counts the codes no sentence of the title or abstract supports.

In [ ]:
if have_data:
    coded = ["practice", "data_origin", "factors", "field", "source_type", "language"]
    display(counts[counts["field"].isin(coded)].reset_index(drop=True))
    display(counts[~counts["field"].isin(coded)].reset_index(drop=True))


### Criteria compared

In [ ]:
if have_data:
    n = len(src)
    by = lambda col: ", ".join(f"{v} {c}" for v, c in src[col].value_counts().items())
    evidence = {
        "source type": by("source_type"),
        "empirical design": f"empirical {int((src['source_type'] == 'empirical').sum())} of {n}",
        "language": by("language"),
        "date": f"2007-2014 {int((src['set'] == 'development').sum())}, 2015-2018 {int((src['set'] == 'test').sum())}",
        "topic": f"practice: {by('practice')}",
        "reuse": f"practice: {by('practice')}",
        "data_origin": by("data_origin"),
        "factors": f"sources with no factor: {int((src['themes'] == '').sum())} of {n}",
        "field and tier": by("field"),
    }
    display(comparison.assign(evidence=comparison["criterion"].map(evidence)))


### Search statements

The four search statements @chawinga2019 report, matched against the titles and abstracts of the reference set, once with each part as a phrase and once as separate words.

In [ ]:
if have_data:
    display(statements.drop(columns=["ref_ids"]))

### Verdict

A PCC part is *covered* when sources eligible apart from year carry it in both sets.

In [ ]:
if have_data:
    display(verdict)

## Step 2. Terms by PCC part

Terms come from the development set: the terms recorded in the coding, and MeSH headings, author keywords and OpenAlex keywords the local model placed under a PCC part. *Sharing* and *reuse* terms go to step 3. Factor terms go to screening. A term from a source coded *both* sits under *sharing* and *reuse*.

In [ ]:
if have_data:
    n = terms.groupby(["pcc_part", "database"]).size().unstack(fill_value=0).reindex(PARTS)
    display(n)

In [ ]:
if have_data:
    sr = terms[(terms["database"] == "PubMed") & terms["pcc_part"].isin(["sharing", "reuse"])].copy()
    sr["count"] = sr["count"].astype(int)
    display(sr.sort_values(["pcc_part", "count"], ascending=[True, False])[["pcc_part", "term", "type", "count"]].reset_index(drop=True))

In [ ]:
if have_data:
    fac = terms[(terms["database"] == "PubMed") & ~terms["pcc_part"].isin(["sharing", "reuse"])]
    display(fac[["pcc_part", "term", "type", "count", "sources_added"]].reset_index(drop=True))

## Term evidence

The query searches titles for a form of *share* within 5 words of *data*, and leaves out titles on blockchain, device networks and cloud computing. It finds 29 of the 35 indexed development studies and, on the test set, 9 of the 10 indexed studies in scope, in about 7,900 records once duplicates are removed. Runs 1 to 9 measure the phrase *"data sharing"* with factor terms from the Candidate terms table; they show what a factor block costs and finds, and they are the evidence for leaving it out. Runs 10 and 11 replace the phrase with a form of *share* near *data* in the title, which finds more development studies in fewer records with no factor terms, so the factor terms guide screening. Runs 12 to 15 measure ways to narrow that block, run 16 tests the result on the test set, and runs 17 and 18 test it on the known-item list of the search strategy page.

This section chooses the terms of one query per database from evidence on the development set. Each run asks two things of a query in PubMed, Scopus and Web of Science. The first is how many records it returns for 2015 to 2025 in English or French, the review's limits. The second is which development studies it finds: the sources eligible apart from year, published 2007 to 2014, that have a DOI or PMID. Those are checked by identifier with no date limit. The test set stays held out until run 16.

The runs query the databases' APIs, so they run on the analysis server from the repository root, with the keys loaded (`source ~/.rds-review-keys`). Rendering skips them (`eval: false`) and shows the files they wrote to `data/raw/search-development/`. To repeat a run, run the setup cell and then the run's cell. The runs follow the order in which each one's result set up the next.

In [ ]:
#| eval: false
# Setup for every run below.
import re
import sys
import itertools

sys.path.insert(0, str(ROOT / "analysis" / "00_search-development" / "retrieval"))
import search_lib as sl

sl.BATCH = 10  # identifiers per request when checking which development studies a query finds

LANG = {"PubMed": "(english[la] OR french[la])", "Scopus": "LANGUAGE(english OR french)",
        "Web of Science": "LA=(English OR French)"}
TAG = {"title": {"PubMed": "{}[ti]", "Scopus": "TITLE({})", "Web of Science": "TI=({})"},
       "anywhere": {"PubMed": "{}[tiab]", "Scopus": "TITLE-ABS-KEY({})", "Web of Science": "TS=({})"}}

# The sharing block that run 1 settled on.
SHARING = {"PubMed": ['"data sharing"', '"shar* data"'],
           "Scopus": ['"data sharing"'],
           "Web of Science": ['"data sharing"']}

# The factor terms of the Candidate terms table, then general words for a factor that the table lacks.
CANDIDATE_TERMS = ["polic*", "challenge*", "repositor*", "ethic*", "privacy", "infrastructure", "barrier*",
                   "perception*", '"open science"', "motivat*", "interoperab*", '"open data"', '"open access"',
                   "consent", '"open research"', "licens*", "attitude*", "incentive*", "mandate*", "willingness",
                   "obstacle*", "stewardship", '"data management plan*"', '"data curation"', "facilitator*",
                   "enabler*", "reluctan*", '"digital curation"']
GENERAL_WORDS = ["factor*", "influenc*", "determinant*", "driver*", "concern*", "trust"]
ALL_FACTORS = CANDIDATE_TERMS + GENERAL_WORDS
ZERO_STUDY = ["facilitator*", "driver*", "interoperab*", '"open research"', '"digital curation"']  # run 5
OUT_OF_SCOPE = ["privacy"]  # run 6
FACTORS = [t for t in ALL_FACTORS if t not in ZERO_STUDY + OUT_OF_SCOPE]


def limit(db):
    return f"{sl.date_limit(db, 2015, 2025)} AND {LANG[db]}"


def field(db, where, terms):
    """The terms OR-joined in one field: the title, or anywhere in title, abstract and keywords."""
    return "(" + " OR ".join(TAG[where][db].format(t) for t in terms) + ")"


def version3(db, factors, near):
    """Data sharing in the title, OR data sharing with a factor term anywhere, or within 5 words with near."""
    v1 = field(db, "title", SHARING[db])
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if not near:
        v2 = f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)})"
    elif db == "Scopus":
        v2 = f"TITLE-ABS-KEY(({s}) W/5 ({f}))"
    else:
        v2 = f"TS=(({s}) NEAR/5 ({f}))"
    return f"({v1} OR {v2})"


FORMS = ("sharing", "shared", "share", "shares")  # PubMed proximity takes no truncation


def share_near_data(db, where, n):
    """A form of share within n words of data, in the title or anywhere. Any word order matches."""
    if db == "PubMed":
        tag = "ti" if where == "title" else "tiab"
        return "(" + " OR ".join(f'"{w} data"[{tag}:~{n}]' for w in FORMS) + ")"
    t = {"Scopus": {"title": "TITLE", "anywhere": "TITLE-ABS-KEY"},
         "Web of Science": {"title": "TI=", "anywhere": "TS="}}[db][where]
    return f"{t}(shar* W/{n} data)" if db == "Scopus" else f"{t}(shar* NEAR/{n} data)"


def limit_near(db):
    """Web of Science's API rejects LA beside shar* NEAR/n data (run 10), so there the date limit stands alone."""
    return sl.date_limit(db, 2015, 2025) if db == "Web of Science" else limit(db)


def fetch(db, q, lim):
    """Every record a query returns, with DOI, title and year."""
    full = f"({q}) AND {lim}"
    if db == "PubMed":
        return list(sl.pubmed_summary(sl.pubmed_all_ids(full)).values())
    return sl.scopus_records(full) if db == "Scopus" else sl.wos_records(full)


def unique_count(recs):
    """Records left once those sharing a DOI, or a title and year, count as one. bib-dedupe finds as many or more."""
    seen, unique = {}, 0
    for r in recs:
        keys = [k for k in (("doi", r["doi"]) if r["doi"] else None,
                            ("title", re.sub(r"[^a-z0-9]", "", r["title"].lower()) + r["year"]) if r["title"] else None)
                if k]
        hit = next((seen[k] for k in keys if k in seen), None)
        if hit is None:
            unique += 1
            hit = unique
        for k in keys:
            seen[k] = hit
    return unique


SRC = sl.read("reference_sources.csv")
DEV = [s for s in SRC if s["set"] == "development" and s["eligible_apart_from_year"] == "yes"]
LABEL = {s["ref_id"]: s["label"] for s in SRC}


def dev_in(db):
    """Development studies a database can be checked for: a DOI, or in PubMed a PMID."""
    return [s for s in DEV if sl.testable(db, s)]

In [ ]:
import re


def show(name, columns=None):
    """Display a file a run wrote, or say the run has not happened."""
    if not path(name).exists():
        print(f"Not run yet. Missing: {name}")
        return None
    t = read(name)
    display(t[columns] if columns else t)
    return t


def unique_records(recs):
    """Records left once those sharing a DOI, or a title and year, count as one. bib-dedupe finds as many or more."""
    seen, unique = {}, 0
    for doi, title, year in recs[["doi", "title", "year"]].itertuples(index=False):
        keys = [k for k in (("doi", doi) if doi else None,
                            ("title", re.sub(r"[^a-z0-9]", "", title.lower()) + year) if title else None) if k]
        hit = next((seen[k] for k in keys if k in seen), None)
        if hit is None:
            unique += 1
            hit = unique
        for k in keys:
            seen[k] = hit
    return unique

### Run 1. Sharing terms

Each candidate term for the sharing part of the PCC table, alone, in title, abstract and keywords: its records, the development studies it finds, how many of those *"data sharing"* misses, and how many no other candidate finds. The table shows whether any term earns a place beside *"data sharing"*.

In PubMed, *"shar\* data"* adds Bertzky and Stoll-Kleemann 2009, Cooper 2007 and Rohlfing and Poline 2012, so the sharing block keeps it there. *"data publication"*, *"data access"* and *"data availability"* each add only Costello 2009, so the block leaves them out. `log.md` records the choice.

In [ ]:
#| eval: false
SHARING_CANDIDATES = ['"data sharing"', '"research data sharing"', '"shar* data"', '"shared data"',
                      '"sharing of data"', '"open data sharing"', '"data access"', '"data availability"',
                      '"data publication"', '"data dissemination"', '"data deposit*"', '"data repositor*"',
                      '"open data"', '"data reuse"', '"data re-use"', '"secondary use"', '"FAIR data"',
                      '"FAIR principles"']
MESH = '"Information Dissemination"[mh:noexp]'  # PubMed only

rows = []
for db in DATABASES:
    dev = dev_in(db)
    terms = SHARING_CANDIDATES + ([MESH] if db == "PubMed" else [])
    found = {}
    for t in terms:
        w = t if t == MESH else TAG["anywhere"][db].format(t)
        found[t] = (sl.count(db, f"({w}) AND {limit(db)}"), sl.retrieved(db, w, dev))
    for t in terms:
        n, hits = found[t]
        others = set().union(*(found[o][1] for o in terms if o != t))
        rows.append(dict(database=db, term="Information Dissemination [MeSH]" if t == MESH else t,
                         records_2015_2025=n, development_found=len(hits),
                         beyond_data_sharing=len(hits - found['"data sharing"'][1]),
                         only_this_term=len(hits - others), limit=limit(db),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing.csv", rows)

In [ ]:
show("term_table_sharing.csv", ["database", "term", "records_2015_2025", "development_found",
                                "beyond_data_sharing", "only_this_term"])

### Runs 2 and 3. Factor terms, one at a time and in small combinations

Run 2 pairs each factor term with the sharing block, anywhere in the record and, in Scopus and Web of Science, within 5 words of it. It then builds a block one term at a time, each time adding the term with the most new studies per extra record. Run 3 searches every combination of up to seven terms offline, from the studies each term finds, and counts the records of the best in each database.

These runs picked a block of a few terms by yield. Read against the PCC table, a block that small looks picked: a reader asks why those terms and no others. Runs 5 to 9 test the whole Candidate terms table in its place, with six general words for a factor added: *factor\**, *influenc\**, *determinant\**, *driver\**, *concern\** and *trust*. Claude proposed the general words; the table has none.

In [ ]:
#| eval: false
def sharing_and(db, factors, near=False):
    """The sharing block AND any factor term: anywhere in the record, or within 5 words."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(factors)
    if db == "PubMed":
        return field(db, "anywhere", SHARING[db]) + (f" AND {field(db, 'anywhere', factors)}" if factors else "")
    if not factors:
        return field(db, "anywhere", SHARING[db])
    if near:
        return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"
    return f"{field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', factors)}"


# Run 2
rows = []
for db in DATABASES:
    dev = dev_in(db)
    base = sl.retrieved(db, sharing_and(db, []), dev)
    for near in ([False] if db == "PubMed" else [False, True]):  # PubMed has no proximity for truncated terms
        mode = "within 5 words" if near else "anywhere in title or abstract"
        found = {f: (sl.count(db, f"({sharing_and(db, [f], near)}) AND {limit(db)}"),
                     sl.retrieved(db, sharing_and(db, [f], near), dev)) for f in ALL_FACTORS}
        for f in ALL_FACTORS:
            rows.append(dict(database=db, placement=mode, step="", term=f, records_2015_2025=found[f][0],
                             development_found=len(found[f][1]), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in found[f][1]))))
        # the block one term at a time: most new studies per extra record first
        chosen, got, load = [], set(), 0
        while True:
            best = None
            for f in ALL_FACTORS:
                new = found[f][1] - got
                if f in chosen or not new:
                    continue
                n = sl.count(db, f"({sharing_and(db, chosen + [f], near)}) AND {limit(db)}")
                if best is None or len(new) / max(n - load, 1) > best[0]:
                    best = (len(new) / max(n - load, 1), f, n, new)
            if best is None:
                break
            _, f, load, new = best
            chosen.append(f)
            got |= new
            rows.append(dict(database=db, placement=mode, step=len(chosen), term=f, records_2015_2025=load,
                             development_found=len(got), of=len(base),
                             studies="; ".join(sorted(LABEL[r] for r in got))))
sl.write("term_table_factors.csv", rows)

# Run 3: a combination's studies are the union of its terms' studies, so they are worked out offline; only the
# record counts of the best combinations go to the databases.
MAX_TERMS, VERIFY = 7, 3
table = [r for r in sl.read("term_table_factors.csv") if r["step"] == ""]
rows = []
for (db, mode), group in itertools.groupby(table, key=lambda r: (r["database"], r["placement"])):
    group = [r for r in group if int(r["development_found"]) > 0]
    near = mode == "within 5 words"
    studies = sorted({s for r in group for s in r["studies"].split("; ")})
    bit = {s: 1 << i for i, s in enumerate(studies)}
    terms = [(r["term"], int(r["records_2015_2025"]), sum(bit[s] for s in r["studies"].split("; ")))
             for r in group]
    best = {}
    for k in range(1, min(MAX_TERMS, len(terms)) + 1):
        for combo in itertools.combinations(terms, k):
            mask = 0
            for _, _, m in combo:
                mask |= m
            # records estimated as the sum of the terms' own records, to rank; the real count comes next
            best.setdefault(bin(mask).count("1"), []).append((sum(c[1] for c in combo), [c[0] for c in combo]))
    for n in sorted(best, reverse=True):
        for _, combo in sorted(best[n])[:VERIFY]:
            rows.append(dict(database=db, placement=mode, studies_kept=n,
                             records_2015_2025=sl.count(db, f"({sharing_and(db, combo, near)}) AND {limit(db)}"),
                             terms=" OR ".join(combo)))
sl.write("term_table_factor_combinations.csv", rows)

In [ ]:
t = show("term_table_factors.csv")

In [ ]:
t = show("term_table_factor_combinations.csv")

### Run 4. Data sharing in the title, or anywhere

Where *"data sharing"* must appear sets the base every factor term combines with, so it comes before any choice of factor terms. The run counts the sharing block in the title alone and anywhere in title, abstract and keywords, and names the development studies only the second finds.

Title only cuts records by about 80% and loses 12 known studies in Scopus and 13 in Web of Science.

In [ ]:
#| eval: false
rows = []
for db in DATABASES:
    for where in ("title", "anywhere"):
        q = field(db, where, SHARING[db])
        hits = sl.retrieved(db, q, dev_in(db))
        rows.append(dict(database=db, field=where, query=q, limit=limit(db),
                         records_2015_2025=sl.count(db, f"{q} AND {limit(db)}"), development_found=len(hits),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_field.csv", rows)

In [ ]:
t = show("term_table_sharing_field.csv", ["database", "field", "records_2015_2025", "development_found"])
if t is not None:
    for db, g in t.groupby("database", sort=False):
        title, anywhere = (set(g.loc[g["field"] == f, "studies"].iloc[0].split("; ")) for f in ("title", "anywhere"))
        print(f"{db}, found anywhere but not in the title: {'; '.join(sorted(anywhere - title))}")

### Run 5. Three versions of the query

Three versions, each with the whole factor block:

1. *"data sharing"* in the title, with no factor terms;
2. *"data sharing"* anywhere AND a factor term anywhere;
3. version 1 OR version 2.

Each of versions 2 and 3 runs once with the full block and once without the five terms that find no development study in any database (*facilitator\**, *driver\**, *interoperab\**, *"open research"*, *"digital curation"*). A query drops out when another finds at least as many development studies in fewer records. The rule needs no target for records or studies.

In every database the full block drops out against the block without the five terms, which add records and no studies, so the five terms go. Versions 1, 2 and 3 survive: each finds more studies than the one before at more records.

In [ ]:
#| eval: false
blocks = {"full": ALL_FACTORS, "full minus zero-study terms": [t for t in ALL_FACTORS if t not in ZERO_STUDY]}
rows = []
for db in DATABASES:
    dev = dev_in(db)
    res = {"1 title only": field(db, "title", SHARING[db])}
    for b, factors in blocks.items():
        res[f"2 anywhere AND factor, {b}"] = (f"({field(db, 'anywhere', SHARING[db])} AND "
                                              f"{field(db, 'anywhere', factors)})")
        res[f"3 version 1 OR version 2, {b}"] = version3(db, factors, near=False)
    res = {k: (q, sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)) for k, q in res.items()}
    for k, (q, n, hits) in res.items():
        beaten = [o for o, (_, n2, h2) in res.items() if o != k and len(h2) >= len(hits) and n2 < n]
        rows.append(dict(database=db, version=k, query=q, limit=limit(db), records_2015_2025=n,
                         development_found=len(hits), of=len(dev), beaten_by="; ".join(beaten),
                         studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_structure.csv", rows)

In [ ]:
t = show("term_table_structure.csv", ["database", "version", "records_2015_2025", "development_found", "beaten_by"])

### Run 6. What each factor term costs

Version 3 with the 29 remaining terms, removing one term at a time and putting it back: the records each term alone adds, and the development studies no other term finds. Every term is measured against the same full block, so the order of removal plays no part.

Only *repositor\** in PubMed (Coady and Wagner 2013) and *challenge\** in Web of Science (Van Horn and Gazzaniga 2013) find a study no other term finds. No single term carries much of the load.

*privacy* goes from the block on two grounds. Data protection falls outside the factors the PCC table asks about, and removing it loses no study in any database.

In [ ]:
#| eval: false
factors = [t for t in ALL_FACTORS if t not in ZERO_STUDY]
rows = []
for db in DATABASES:
    dev = dev_in(db)
    q = version3(db, factors, near=False)
    n0, h0 = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
    for t in factors:
        q = version3(db, [f for f in factors if f != t], near=False)
        n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
        lost = sorted(LABEL[r] for r in h0 - h)
        rows.append(dict(database=db, term_removed=t, records_with_all=n0, records_saved=n0 - n,
                         studies_with_all=len(h0), studies_lost=len(lost), lost="; ".join(lost)))
sl.write("term_table_leave_one_out.csv", rows)

In [ ]:
t = show("term_table_leave_one_out.csv")

### Run 7. Removing terms one after another, and proximity

Version 3 with the 28 terms, removing one term per step without putting it back, in the order of run 6: terms that lose no study first, most records saved first. The order is fixed before the run. Each step records the records left and the studies lost so far. In Scopus and Web of Science the run repeats with the factor term within 5 words of *"data sharing"*, which requires the factor word to be about data sharing.

Studies drop out by the second or third step in Scopus and Web of Science: studies several terms find are lost once all of them go. With one factor term left, version 3 still returns 2,209 records in PubMed, 5,632 in Scopus and 9,051 in Web of Science, so cutting terms cannot bring the load down far. The order also removes *polic\** and *barrier\** early, because they cost the most records, and a query about factors in data sharing without them makes little sense to a reader. Proximity makes the large difference: with all 28 terms, 9,018 records in Scopus against 21,664, and 7,175 in Web of Science against 15,294.

In [ ]:
#| eval: false
loo = sl.read("term_table_leave_one_out.csv")
rows = []
for db in DATABASES:
    dev = dev_in(db)
    order = [r["term_removed"] for r in sorted((r for r in loo if r["database"] == db),
             key=lambda r: (int(r["studies_lost"]), -int(r["records_saved"])))
             if r["term_removed"] not in OUT_OF_SCOPE]
    for near in ([False] if db == "PubMed" else [False, True]):
        kept = list(order)
        h_start = sl.retrieved(db, version3(db, kept, near), dev)
        for step in range(len(order)):
            cut = order[step - 1] if step else ""
            if cut:
                kept.remove(cut)
            q = version3(db, kept, near)
            n, h = sl.count(db, f"{q} AND {limit(db)}"), sl.retrieved(db, q, dev)
            rows.append(dict(database=db, placement="within 5 words" if near else "anywhere", step=step,
                             term_cut=cut, terms_left=len(kept), records_2015_2025=n, development_found=len(h),
                             of=len(dev), lost_so_far="; ".join(sorted(LABEL[r] for r in h_start - h)),
                             query=q, limit=limit(db)))
sl.write("term_table_cuts.csv", rows)

In [ ]:
t = show("term_table_cuts.csv", ["database", "placement", "step", "term_cut", "terms_left", "records_2015_2025",
                                 "development_found", "lost_so_far"])

### Run 8. Version 3's records

Version 3 with the 28 terms: anywhere in PubMed, within 5 words in Scopus and Web of Science. The run names the development studies proximity loses against anywhere in Scopus and Web of Science, then fetches every record the three queries return and saves them.

The count below treats records that share a DOI, or a title and year, as one. It estimates the load. bib-dedupe, which deduplicates the review's records, compares more fields and will find as many duplicates or more.

In [ ]:
#| eval: false
for db in ["Scopus", "Web of Science"]:
    lost = sl.retrieved(db, version3(db, FACTORS, False), dev_in(db)) - sl.retrieved(db, version3(db, FACTORS, True), dev_in(db))
    print(f"{db}, found anywhere but not within 5 words: {'; '.join(sorted(LABEL[r] for r in lost)) or '-'}")

recs = []
for db in DATABASES:
    q = f"{version3(db, FACTORS, near=db != 'PubMed')} AND {limit(db)}"
    if db == "PubMed":
        got = list(sl.pubmed_summary(sl.pubmed_all_ids(q)).values())
    elif db == "Scopus":
        got = sl.scopus_records(q)
    else:
        got = sl.wos_records(q)
    recs += [dict(r, database=db) for r in got]
sl.write("final_query_records.csv", recs)

In [ ]:
if not path("final_query_records.csv").exists():
    print("Not run yet. Missing: final_query_records.csv")
else:
    recs = read("final_query_records.csv")
    display(recs.groupby("database", sort=False).size().rename("records").to_frame())
    print(f"All three databases: {len(recs)} records, {unique_records(recs)} after removing duplicates "
          f"by DOI, or by title and year")

### Run 9. Development studies version 3 misses

Version 3 with the 28 terms, in each database, against the development studies by identifier. A study counts as found when any of the three databases finds it. For each study none finds, the file gives its full record and, per database, whether the database does not index it or indexes it and the query misses it. Studies with no DOI or PMID cannot be checked.

Of the 40 development studies with a DOI or PMID, the three databases together find 28. Five are in none of the databases. Seven are indexed and missed: Borgman 2012, Cohn 2012, Costello 2009, Haddow et al. 2011, McCullough 2009, Parr 2007 and Zimmerman 2008. Of those seven, matching anywhere would find only Borgman in place of proximity.

In [ ]:
#| eval: false
found, indexed = {}, {}
for db in DATABASES:
    found[db] = sl.retrieved(db, version3(db, FACTORS, near=db != "PubMed"), dev_in(db))
    indexed[db] = {s["ref_id"] for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
anyf = set().union(*found.values())

rows = []
for s in sorted(DEV, key=lambda s: s["label"]):
    if s["ref_id"] in anyf:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    rows.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                     pmid=s["pmid"], field=s["field"], source_type=s["source_type"],
                     pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("final_query_missed.csv", rows)

In [ ]:
t = show("final_query_missed.csv")

### Run 10. A form of share near data, in place of the phrase

Of the seven indexed development studies version 3 misses in run 9, three word data sharing in ways the phrase *"data sharing"* does not match: Borgman 2012 ("sharing research data"), Haddow et al. 2011 ("sharing of health data") and Zimmerman 2008 ("sharing and reuse of ecological data"). The run compares the sharing block as the phrase with a form of *share* within 3 and within 5 words of *data*, in the title and anywhere. PubMed's proximity search takes no truncation, so there the block writes out *sharing*, *shared*, *share* and *shares*.

In the title, a form of *share* within 3 words of *data* finds 26 development studies in 8,725 records in Scopus and 27 in 7,148 in Web of Science, against 21 for version 3 at about the same records, with no factor terms at all. Anywhere in the record, the same block returns 50,000 to 80,000 records and drops out.

Web of Science's API rejects the language limit `LA` beside *shar\* NEAR/n data*, with an error that says the API does not search the tag at all. A check showed the limit works elsewhere: an invented language returns no records, and English or French removes about 0.5%. Web of Science counts for share-near-data queries therefore carry no language limit. The review's exports come from the Web of Science site, where the limit works.

In [ ]:
#| eval: false
rows = []
for db in DATABASES:
    dev = dev_in(db)
    phrase = {}
    for where in ("title", "anywhere"):
        for n in (0, 3, 5):
            q = field(db, where, SHARING[db]) if n == 0 else share_near_data(db, where, n)
            lim = limit(db) if n == 0 else limit_near(db)
            recs, hits = sl.count(db, f"({q}) AND {lim}"), sl.retrieved(db, q, dev)
            if n == 0:
                phrase[where] = hits
            rows.append(dict(database=db, field=where, sharing="phrase" if n == 0 else f"within {n} words",
                             query=q, limit=lim, records_2015_2025=recs, development_found=len(hits),
                             beyond_phrase="; ".join(sorted(LABEL[r] for r in hits - phrase[where])),
                             studies="; ".join(sorted(LABEL[r] for r in hits))))
sl.write("term_table_sharing_proximity.csv", rows)

In [ ]:
t = show("term_table_sharing_proximity.csv", ["database", "field", "sharing", "records_2015_2025",
                                              "development_found", "beyond_phrase"])

### Run 11. Three candidate queries across the three databases

Three candidates, each judged across the three databases together: the development studies at least one database finds, out of the 35 that at least one indexes, and the records with duplicates removed.

1. a form of *share* within 3 words of *data*, in the title;
2. the same within 5 words;
3. candidate 2 OR the factor part of version 3: *"data sharing"* with any of the 28 factor terms, anywhere in PubMed and within 5 words in Scopus and Web of Science.

| | Records | Without duplicates | Development studies found |
|---|--:|--:|--:|
| Version 3 (runs 8 and 9) | 23,488 | 14,255 | 28 of 35 |
| 1. within 3 words, title | 18,258 | 9,591 | 28 of 35 |
| 2. within 5 words, title | 19,892 | 10,493 | 29 of 35 |
| 3. candidate 2 OR factor part | 30,985 | 18,135 | 31 of 35 |

Candidate 1 finds what version 3 finds in about 4,700 fewer records, so version 3 drops out. Candidates 1, 2 and 3 survive the rule of run 5. Candidate 2 adds Zimmerman 2008 for 902 records; candidate 3 adds Fisher and Fortmann 2010 and Piwowar 2011 for 7,642 more. Cohn 2012, Costello 2009, McCullough 2009 and Parr 2007 stay missed by every query tested: their titles speak of open sourcing, online publication, open access journals and DataONE, and none of them says sharing data.

In [ ]:
#| eval: false
in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}


def factor_part(db):
    """The factor part of version 3: anywhere in PubMed, within 5 words in Scopus and Web of Science."""
    s, f = " OR ".join(SHARING[db]), " OR ".join(FACTORS)
    if db == "PubMed":
        return f"({field(db, 'anywhere', SHARING[db])} AND {field(db, 'anywhere', FACTORS)})"
    return f"TITLE-ABS-KEY(({s}) W/5 ({f}))" if db == "Scopus" else f"TS=(({s}) NEAR/5 ({f}))"


CANDIDATES = {"1 title, within 3 words": lambda db: share_near_data(db, "title", 3),
              "2 title, within 5 words": lambda db: share_near_data(db, "title", 5),
              "3 version 2 OR factor part": lambda db: f"({share_near_data(db, 'title', 5)} OR {factor_part(db)})"}
summary, all_recs = [], []
for name, make in CANDIDATES.items():
    found, recs = set(), []
    for db in DATABASES:
        q = make(db)
        hits = sl.retrieved(db, q, dev_in(db))
        got = fetch(db, q, limit_near(db))
        recs += [dict(r, database=db, candidate=name) for r in got]
        found |= hits
        summary.append(dict(candidate=name, database=db, query=q, limit=limit_near(db), records_2015_2025=len(got),
                            development_found=len(hits), studies="; ".join(sorted(LABEL[r] for r in hits))))
    summary.append(dict(candidate=name, database="all three", records_2015_2025=len(recs),
                        unique_records=unique_count(recs), development_found=len(found), of_indexed=len(in_any),
                        missed_indexed="; ".join(sorted(LABEL[r] for r in in_any - found))))
    all_recs += recs
sl.write("candidate_queries.csv", summary)
sl.write("candidate_records.csv", all_recs)

In [ ]:
t = show("candidate_queries.csv", ["candidate", "database", "records_2015_2025", "unique_records",
                                   "development_found", "of_indexed", "missed_indexed"])

### Run 12. Document types

Candidate 2's records by each database's document type, and the database's type for each development study candidate 2 finds, beside the source type the coding gave it. The run tests whether a document type limit could narrow the block. The protocol makes all source types eligible, so any limit would change the eligibility criteria.

Editorials, letters, notes and news make up about 18% of the records in PubMed, 8% in Scopus and 11% in Web of Science. The databases' types do not follow the coding's source types: most opinion pieces are indexed as Article or Review, and a Note or Editorial can be a policy document (Harding et al. 2013) or an empirical study (Huang et al. 2013). A limit would leave most opinion pieces in and drop sources the review keeps. Errata, corrections and retractions, about 250 records, are no source type at all. Conference papers make up 34% of Scopus's records and meeting items 24% of Web of Science's.

In [ ]:
#| eval: false
import os
import collections

Q2 = {db: share_near_data(db, "title", 5) for db in DATABASES}
rows = []


def record(db, counts, total, known):
    for t, n in counts.items():
        rows.append(dict(database=db, what="records", document_type=t, n=n, of=total))
    for r, t in known.items():
        rows.append(dict(database=db, what="development study", document_type=t, label=LABEL[r],
                         source_type=next(s["source_type"] for s in DEV if s["ref_id"] == r)))


# PubMed: publication types overlap, so each count is the records carrying that type
db, q = "PubMed", f"({Q2['PubMed']}) AND {limit('PubMed')}"
pm_types = ["Journal Article", "Review", "Editorial", "Comment", "Letter", "News", "Published Erratum",
            "Practice Guideline", "Guideline", "Systematic Review", "Meta-Analysis"]
counts = {t: sl.count(db, f'{q} AND "{t}"[pt]') for t in pm_types}
counts["none of Editorial, Comment, Letter, News"] = sl.count(db, f"{q} NOT (Editorial[pt] OR Comment[pt] OR Letter[pt] OR News[pt])")
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    _, ids = sl.pubmed_search(sl.id_clause(db, [s for s in DEV if s["ref_id"] == r]), retmax=1)
    j = sl._get(sl.EU + "esummary.fcgi", dict(db="pubmed", id=ids[0], retmode="json",
                api_key=os.environ["NCBI_API_KEY"]), pause=0.11).json()["result"][ids[0]]
    known[r] = ", ".join(j.get("pubtype", []))
record(db, counts, sl.count(db, q), known)

# Scopus: one document type per record
db, q = "Scopus", f"({Q2['Scopus']}) AND {limit('Scopus')}"
sc_types = {"ar": "Article", "re": "Review", "ed": "Editorial", "le": "Letter", "no": "Note", "sh": "Short survey",
            "cp": "Conference paper", "ch": "Book chapter", "bk": "Book", "er": "Erratum", "cr": "Conference review",
            "dp": "Data paper"}
counts = {name: sl.count(db, f"{q} AND DOCTYPE({code})") for code, name in sc_types.items()}
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    j = sl._get(sl.SC, dict(query=f"DOI({doi})", field="doi,subtypeDescription"),
                {"X-ELS-APIKey": os.environ["SCOPUS_API_KEY"], "Accept": "application/json"}, pause=0.12).json()
    known[r] = j["search-results"]["entry"][0].get("subtypeDescription", "?")
record(db, counts, sl.count(db, q), known)

# Web of Science: types read from every record; a record can carry more than one
db, q = "Web of Science", f"({Q2['Web of Science']}) AND {limit_near('Web of Science')}"
counts, total, page = collections.Counter(), 0, 1
while True:
    j = sl.wos_search(q, limit=50, page=page)
    for h in j.get("hits", []):
        counts.update(h.get("types", ["?"]))
        total += 1
    if page * 50 >= j["metadata"]["total"] or not j.get("hits"):
        break
    page += 1
known = {}
for r in sl.retrieved(db, Q2[db], dev_in(db)):
    doi = next(s["doi"] for s in DEV if s["ref_id"] == r)
    h = sl.wos_search(f"DO=({doi})", limit=1)["hits"]
    known[r] = ", ".join(h[0].get("types", ["?"])) if h else "?"
record(db, dict(counts), total, known)
sl.write("document_types.csv", rows)

In [ ]:
t = show("document_types.csv")

### Run 13. Data sharing in the computing sense

Many titles that put *share* near *data* concern computing: encryption schemes, blockchain frameworks, cloud storage. The run counts candidate 2's records whose title also carries one of 16 technical words, the development studies among them, and each word's records, and prints 25 Scopus titles to judge whether the words pick out the computing sense.

The technical words mark 6% of the records in PubMed, 35% in Scopus and 29% in Web of Science, and no development study. The sample titles are almost all technical. A few concern research data: "Blockchain Technology for Big-data Sharing in Material Genome Engineering". *scheme* also catches governance and funding schemes, *secure* catches secure access to research data, and *federated learning* is an alternative to sharing health research data that the review may cover, so these three leave the list.

In [ ]:
#| eval: false
import os

TECH_WORDS = ["encrypt*", "cryptograph*", "blockchain*", "cloud", '"access control"', "attribute-based",
              '"privacy-preserving"', '"privacy preserving"', '"smart contract*"', '"internet of things"', "IoT",
              "vehicular", '"edge computing"', '"federated learning"', "secure", "scheme"]
rows = []
for db in DATABASES:
    q, lim = f"({share_near_data(db, 'title', 5)})", limit_near(db)
    total = sl.count(db, f"{q} AND {lim}")
    any_tech = field(db, "title", TECH_WORDS)
    lost = sl.retrieved(db, f"{q} AND {any_tech}", dev_in(db))
    rows.append(dict(database=db, term="any", records=sl.count(db, f"{q} AND {any_tech} AND {lim}"), of=total,
                     development_studies="; ".join(sorted(LABEL[r] for r in lost))))
    for t in TECH_WORDS:
        rows.append(dict(database=db, term=t, records=sl.count(db, f"{q} AND {field(db, 'title', [t])} AND {lim}"),
                         of=total))
    if db == "Scopus":  # titles to judge whether the words pick out the computing sense
        j = sl._get(sl.SC, dict(query=f"{q} AND {any_tech} AND {lim}", count=25, field="dc:title"),
                    {"X-ELS-APIKey": os.environ["SCOPUS_API_KEY"], "Accept": "application/json"}).json()
        for e in j["search-results"].get("entry", []):
            print(e.get("dc:title", ""))
sl.write("technical_sense.csv", rows)

In [ ]:
t = show("technical_sense.csv")

### Run 14. Candidate 2 without the computing sense

Candidate 2 with a NOT clause on the 13 technical words that remain: *blockchain\**, *cloud*, *encrypt\**, *"privacy-preserving"*, *"privacy preserving"*, *IoT*, *"access control"*, *attribute-based*, *"internet of things"*, *vehicular*, *cryptograph\**, *"edge computing"* and *"smart contract\*"*. Filters that exclude an unwanted sense of a term are common in search strategies; this one rests on the development studies it keeps, the titles it removes, and the test set. The run fetches the records the clause keeps and removes in each database, counts them with duplicates removed, checks the development studies, and saves every removed title that mentions research, health or science, so the borderline cases can be read.

The clause cuts the records from 10,493 to 7,505 once duplicates are removed and keeps all 29 development studies. But 698 of the removed titles mention research, health or science, and some of them concern the factors the review maps: "Role of Incentives in the Use of Blockchain-Based Platforms for Sharing Sensitive Health Data: Experimental Study", "Is blockchain the breakthrough we are looking for to facilitate genomic data sharing? The European Union perspective", "Privacy-preserving data sharing infrastructures for medical research". Blockchain, the Internet of Things and most of this literature came after 2014, so the development set cannot speak to these words: no development study lost says nothing about them.

In [ ]:
#| eval: false
TECH = ["blockchain*", "cloud", "encrypt*", '"privacy-preserving"', '"privacy preserving"', "IoT",
        '"access control"', "attribute-based", '"internet of things"', "vehicular", "cryptograph*",
        '"edge computing"', '"smart contract*"']
RESEARCH = re.compile(r"research|clinical|health|medical|biomedical|genom|patient|scien|trial|cohort|"
                      r"hospital|epidemiolog|biobank|neuro", re.I)


def without_tech(db):
    q, t = share_near_data(db, "title", 5), field(db, "title", TECH)
    return f"(({q}) AND NOT {t})" if db == "Scopus" else f"(({q}) NOT {t})"


in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
kept_recs, removed_recs, found = [], [], set()
for db in DATABASES:
    found |= sl.retrieved(db, without_tech(db), dev_in(db))
    kept_recs += [dict(r, database=db, status="kept") for r in fetch(db, without_tech(db), limit_near(db))]
    removed_recs += [dict(r, database=db, status="removed") for r in
                     fetch(db, f"(({share_near_data(db, 'title', 5)}) AND {field(db, 'title', TECH)})", limit_near(db))]
print(f"{len(kept_recs)} kept, {unique_count(kept_recs)} without duplicates; "
      f"{len(found)} of {len(in_any)} indexed development studies found")

seen, flagged = set(), []
for r in removed_recs:  # removed titles that look like research data, one per title
    key = re.sub(r"[^a-z0-9]", "", r["title"].lower())
    if r["title"] and RESEARCH.search(r["title"]) and key not in seen:
        seen.add(key)
        flagged.append(r)
sl.write("technical_filter_records.csv", kept_recs + removed_recs)
sl.write("technical_filter_flagged.csv", flagged)

In [ ]:
if not path("technical_filter_records.csv").exists():
    print("Not run yet. Missing: technical_filter_records.csv")
else:
    recs = read("technical_filter_records.csv")
    display(recs.groupby(["database", "status"], sort=False).size().unstack())
    for status in ("kept", "removed"):
        print(f"{status}: {unique_records(recs[recs['status'] == status])} without duplicates")
    t = show("technical_filter_flagged.csv", ["database", "title", "year"])

### Run 15. A NOT clause on systems, products and services

The eligibility criteria gain one line: sources that design, build or evaluate a specific system, product or service for sharing data are out of scope. The NOT clause takes only title words that almost always mark such a source, so that it removes before screening what screening would exclude. The run adds the words in four groups, each on top of the last.

| NOT clause | Records without duplicates | Saved by the step | Development studies found |
|---|--:|--:|--:|
| none | 10,493 | | 29 of 35 |
| *blockchain\**, *"smart contract\*"* | 9,157 | 1,336 | 29 |
| + *IoT*, *"internet of things"*, *vehicular*, *"edge computing"* | 8,848 | 309 | 29 |
| + *cloud* | 7,933 | 915 | 29 |
| + *attribute-based*, *cryptograph\** | 7,857 | 76 | 29 |

The clause stops at *cloud*. *attribute-based* and *cryptograph\** save 76 records, too few for two more words a reader must parse; screening excludes such records under the same line. *encrypt\**, *"access control"*, *"privacy-preserving"*, *scheme* and *secure* also mark methods and governance of research data, and *federated learning* marks an alternative to sharing health research data, so all of them stay out of the clause. A study of why people use such a system, as in the incentives study above, concerns factors and would be in scope; the clause removes it, and citation searching is the backstop.

In [ ]:
#| eval: false
BLOCKCHAIN = ["blockchain*", '"smart contract*"']
DEVICES = ["IoT", '"internet of things"', "vehicular", '"edge computing"']
STEPS = [("blockchain", BLOCKCHAIN), ("+ devices", BLOCKCHAIN + DEVICES),
         ("+ cloud", BLOCKCHAIN + DEVICES + ["cloud"]),
         ("+ attribute-based, cryptograph*", BLOCKCHAIN + DEVICES + ["cloud", "attribute-based", "cryptograph*"])]


def share_not(db, terms):
    """A form of share within 5 words of data in the title, NOT any of the terms in the title."""
    q, t = share_near_data(db, "title", 5), field(db, "title", terms)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


in_any = {s["ref_id"] for db in DATABASES for s in dev_in(db) if sl.count(db, sl.id_clause(db, [s])) > 0}
rows, summary = [], []
for name, terms in STEPS:
    recs, found = [], set()
    for db in DATABASES:
        q = share_not(db, terms)
        found |= sl.retrieved(db, q, dev_in(db))
        got = fetch(db, q, limit_near(db))
        recs += [dict(r, database=db, version=name) for r in got]
        summary.append(dict(version=name, database=db, query=q, limit=limit_near(db), records_2015_2025=len(got)))
    summary.append(dict(version=name, database="all three", records_2015_2025=len(recs), unique_records=unique_count(recs),
                        development_found=len(found), of_indexed=len(in_any),
                        missed_indexed="; ".join(sorted(LABEL[r] for r in in_any - found))))
    rows += recs
sl.write("query_exclusion.csv", summary)
sl.write("query_exclusion_records.csv", rows)

In [ ]:
t = show("query_exclusion.csv", ["version", "database", "records_2015_2025", "unique_records", "development_found", "of_indexed"])

### Run 16. The test set

The query, with the NOT clause up to *cloud*, against the test set: the 34 sources eligible apart from year and published from 2015, held out from every run before this one. Each database's coverage carries a Wilson 95% interval. The test set's misses had been read on 16 November, in the step 3 run with the yield rule, before this query existed; the query was not built from them.

| | Test studies found | 95% interval |
|---|--:|---|
| PubMed | 8 of 12 | 0.39 to 0.86 |
| Scopus | 16 of 24 | 0.47 to 0.82 |
| Web of Science | 12 of 19 | 0.41 to 0.81 |
| All three together | 18 of 26 | 0.50 to 0.84 |
| In scope | 9 of 10 | |

Of the 16 test studies the query misses, 8 cannot be found by any query: 3 are in none of the databases, among them two with data sharing in the title (Bishoff and Johnston 2015, He 2016), and 5 have no DOI or PMID. The 8 the query misses in a database name research data management (Chigwada et al. 2017, Chiware and Mathe 2016, Schöpfel et al. 2018, Wittenberg and Elings 2017, Specht et al. 2015), open data or open science (Schmidt et al. 2016, Watson 2015) or reuse (Curty et al. 2017). Only Chiware and Mathe 2016 is in scope; it is added to the search results by hand. Titles naming research data management or open data would add mostly sources out of scope, and open data titles include a literature on open government data.

In [ ]:
#| eval: false
QUERY_NOT = BLOCKCHAIN + DEVICES + ["cloud"]
test = [s for s in SRC if s["set"] == "test" and s["eligible_apart_from_year"] == "yes"]
found, indexed, rows = {}, {}, []
for db in DATABASES:
    t = [s for s in test if sl.testable(db, s)]
    indexed[db] = {s["ref_id"] for s in t if sl.count(db, sl.id_clause(db, [s])) > 0}
    found[db] = sl.retrieved(db, share_not(db, QUERY_NOT), t)
    p, lo, hi = sl.wilson(len(found[db]), len(indexed[db]))
    rows.append(dict(database=db, query=share_not(db, QUERY_NOT), found=len(found[db]), indexed=len(indexed[db]),
                     coverage=p, low=lo, high=hi))
any_found, in_any = set().union(*found.values()), set().union(*indexed.values())
p, lo, hi = sl.wilson(len(any_found), len(in_any))
rows.append(dict(database="all three", found=len(any_found), indexed=len(in_any), coverage=p, low=lo, high=hi))
missed = []
for s in sorted(test, key=lambda s: s["label"]):
    if s["ref_id"] in any_found:
        continue
    where = {db: ("no DOI or PMID" if not (s["doi"] or s["pmid"]) else
                  "not indexed" if s["ref_id"] not in indexed[db] else "indexed, not found") for db in DATABASES}
    missed.append(dict(ref_id=s["ref_id"], label=s["label"], title=s["title"], year=s["year"], doi=s["doi"],
                       pmid=s["pmid"], field=s["field"], source_type=s["source_type"], in_scope=s["in_scope"],
                       pubmed=where["PubMed"], scopus=where["Scopus"], web_of_science=where["Web of Science"]))
sl.write("test_coverage.csv", rows)
sl.write("test_missed.csv", missed)

In [ ]:
t = show("test_coverage.csv", ["database", "found", "indexed", "coverage", "low", "high"])
t = show("test_missed.csv", ["label", "title", "in_scope", "pubmed", "scopus", "web_of_science"])

### Run 17. Known items

The query against the known-item list on the [search strategy](../../materials/search-strategy.qmd) page, chosen before the search was developed: 11 health research sources, 5 cross-disciplinary reviews, 3 adjacent sources and 2 comparators, checked by DOI in each database.

The query finds 8 of the 11 health research sources, all 5 reviews, and all adjacent and comparator sources. It misses three the databases hold. Hutchings et al. 2020 puts seven words between *sharing* and *data*. Swales et al. 2024 (a data transfer agreement) and Inau et al. 2023 (health data stewardship) concern governance mechanisms whose titles do not name sharing.

In [ ]:
#| eval: false
QUERY = {db: share_not(db, QUERY_NOT) for db in DATABASES}  # run 15's NOT clause up to cloud
ITEMS = [("tier 1", "Rowhani-Farid et al. 2017", "10.1186/s41073-017-0028-9"),
         ("tier 1", "Devriendt et al. 2021", "10.1089/bio.2020.0037"),
         ("tier 1", "Kalkman et al. 2019", "10.1186/s12910-019-0359-9"),
         ("tier 1", "Hamilton et al. 2023", "10.1136/bmj-2023-075767"),
         ("tier 1", "Hutchings et al. 2020", "10.1186/s13643-020-01485-5"),
         ("tier 1", "Saulnier et al. 2019", "10.1038/s41597-019-0310-4"),
         ("tier 1", "Gudi et al. 2022", "10.2196/33591"),
         ("tier 1", "Swales et al. 2024", "10.3389/fphar.2024.1332700"),
         ("tier 1", "Andanda and Mlotshwa 2024", "10.1177/17470161241261907"),
         ("tier 1", "Inau et al. 2023", "10.2196/45013"),
         ("tier 1", "Scheibner et al. 2023", "10.1007/s10916-023-02009-z"),
         ("tier 2", "Fecher et al. 2015", "10.1371/journal.pone.0118053"),
         ("tier 2", "Zuiderwijk et al. 2020", "10.1371/journal.pone.0239283"),
         ("tier 2", "Perrier et al. 2020", "10.1371/journal.pone.0229182"),
         ("tier 2", "Chawinga and Zinn 2019", "10.1016/j.lisr.2019.04.004"),
         ("tier 2", "Thøgersen and Borlund 2022", "10.1108/JD-01-2021-0015"),
         ("adjacent", "Alam et al. 2024", "10.1371/journal.pdig.0000652"),
         ("adjacent", "Baines et al. 2024", "10.2196/50421"),
         ("adjacent", "Cascini et al. 2024", "10.1016/j.eclinm.2024.102551"),
         ("comparator", "van Panhuis et al. 2014", "10.1186/1471-2458-14-1144"),
         ("comparator", "Yehudi et al. 2025", "10.1017/dap.2024.79")]
items = [dict(ref_id=f"k{i}", group=g, label=l, doi=d.lower(), pmid="") for i, (g, l, d) in enumerate(ITEMS)]
for db in DATABASES:
    found = sl.retrieved(db, QUERY[db], items)
    for it in items:
        it[db] = ("found" if it["ref_id"] in found else
                  "missed" if sl.count(db, sl.id_clause(db, [it])) > 0 else "not indexed")
sl.write("known_item_test.csv", [dict(group=it["group"], label=it["label"], doi=it["doi"],
                                      found_any=any(it[db] == "found" for db in DATABASES),
                                      pubmed=it["PubMed"], scopus=it["Scopus"], web_of_science=it["Web of Science"])
                                 for it in items])

In [ ]:
t = show("known_item_test.csv")

### Run 18. Governance terms

Two of the known items the query misses concern governance mechanisms that the PCC concept names. The run measures five title terms for them, each with the NOT clause, against the query's 7,933 records: the records each adds once duplicates are removed, and the known sources from 2007 to 2014 and known items it finds that the query misses. The known sources from 2015 stay out, since they have already tested the query.

| Title term | Records added | Known sources added |
|---|--:|---|
| *"data access"* | 1,077 | none |
| *"data transfer"* | 1,617 | Swales et al. 2024 |
| *"data governance"* | 1,120 | none |
| *"data stewardship"* | 82 | Inau et al. 2023 |
| *"FAIR data"*, *"FAIR principles"*, *"FAIR guiding principles"* | 425 | none |
| All five | 4,304 | |

None enters the query. A term added to catch one known source looks picked, and the query stays one idea: a title that names sharing data. The four known sources the query misses that meet the criteria, Chiware and Mathe 2016, Hutchings et al. 2020, Swales et al. 2024 and Inau et al. 2023, are added to the records by hand. This run drew on the known-item list, so that list no longer tests these terms independently.

In [ ]:
#| eval: false
TERMS = {'"data access"': ['"data access"'], '"data transfer"': ['"data transfer"'],
         '"data governance"': ['"data governance"'], '"data stewardship"': ['"data stewardship"'],
         "FAIR": ['"FAIR data"', '"FAIR principles"', '"FAIR guiding principles"']}


def not_tech(db, q):
    t = field(db, "title", QUERY_NOT)
    return f"({q} AND NOT {t})" if db == "Scopus" else f"({q} NOT {t})"


known = [dict(ref_id=f"k{i}", label=r["label"], doi=r["doi"], pmid="")
         for i, r in enumerate(sl.read("known_item_test.csv")) if r["group"] in ("tier 1", "tier 2")]
label = LABEL | {k["ref_id"]: k["label"] for k in known}
base_recs = [r for r in sl.read("query_exclusion_records.csv") if r["version"] == "+ cloud"]
base_u = unique_count(base_recs)
base_found = set().union(*(sl.retrieved(db, QUERY[db], dev_in(db)) | sl.retrieved(db, QUERY[db], known)
                           for db in DATABASES))
rows, all_new = [], []
for name, terms in TERMS.items():
    recs, found = [], set()
    for db in DATABASES:
        q = not_tech(db, field(db, "title", terms))
        found |= sl.retrieved(db, q, dev_in(db)) | sl.retrieved(db, q, known)
        recs += [dict(r, database=db, term=name) for r in fetch(db, q, limit_near(db))]
    rows.append(dict(term=name, records_added=unique_count(base_recs + recs) - base_u,
                     known_added="; ".join(sorted(label[r] for r in found - base_found))))
    all_new += recs
rows.append(dict(term="all five", records_added=unique_count(base_recs + all_new) - base_u))
sl.write("governance_terms.csv", rows)
sl.write("governance_terms_records.csv", all_new)

In [ ]:
t = show("governance_terms.csv")

### The query

```
Title: share* NEAR/5 data
NOT
Title: blockchain* OR "smart contract*" OR IoT OR "internet of things" OR vehicular OR "edge computing" OR cloud

Limits: published 2015 to 2025; English or French
```

- **PubMed:** `("sharing data"[ti:~5] OR "shared data"[ti:~5] OR "share data"[ti:~5] OR "shares data"[ti:~5]) NOT (blockchain*[ti] OR "smart contract*"[ti] OR IoT[ti] OR "internet of things"[ti] OR vehicular[ti] OR "edge computing"[ti] OR cloud[ti])`
- **Scopus:** `TITLE(shar* W/5 data) AND NOT (TITLE(blockchain*) OR TITLE("smart contract*") OR TITLE(IoT) OR TITLE("internet of things") OR TITLE(vehicular) OR TITLE("edge computing") OR TITLE(cloud))`
- **Web of Science:** `TI=(shar* NEAR/5 data) NOT (TI=(blockchain*) OR TI=("smart contract*") OR TI=(IoT) OR TI=("internet of things") OR TI=(vehicular) OR TI=("edge computing") OR TI=(cloud))`

PubMed's proximity search takes no truncation, so its block writes out four forms of *share*. Reuse, the other concept part of the PCC table, is left out: the review asks about sharing.

## Step 3, first version: the yield rule

The first version of step 3 chose terms from `terms.csv` by a yield rule and tested coverage and precision. Its queries returned 8,824, 33,089 and 21,721 records for 2015 to 2025, far more than can be screened, and the Term evidence section above replaced it. Its files stay in the repository, and the cells below show them as a record.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(p, 3), round(max(0, c - h), 3), round(min(1, c + h), 3)


if not have_tests:
    print("Not run yet. Missing: query_tests.csv")
else:
    tests = read("query_tests.csv")
    display(tests.set_index("database").drop(columns=["final_query"]).T)
    for _, t in tests.iterrows():
        print(f"{t['database']}: {t['final_query']}")

In [ ]:
if have_tests:
    if not path("term_results.csv").exists():
        print("Not run yet. Missing: term_results.csv")
    else:
        res = read("term_results.csv")
        display(res.groupby(["database", "status"]).size().unstack(fill_value=0))
        inq = res[res["status"].isin(["in query", "fallback in query"])]
        display(inq[["database", "term", "type", "development_yield", "load_2015_2025", "sources_added"]].reset_index(drop=True))


### Misses

Indexed test set sources the query misses, and development set sources reached only by terms excluded by yield.

In [ ]:
if have_tests and path("query_misses.csv").exists():
    display(read("query_misses.csv"))

### Precision

Precision counts *undecidable* records once as included and once as left out.

In [ ]:
if have_tests:
    rows = []
    for db, slug in DATABASES.items():
        p = path(f"precision_sample_{slug}.csv")
        if not p.exists():
            continue
        s = read(p.name)
        d = s["decision"].str.lower()
        if (d == "").any():
            print(f"{db}: {int((d == '').sum())} of {len(s)} sampled records not screened yet.")
            continue
        inc, exc, und = int((d == "include").sum()), int((d == "exclude").sum()), int((d == "undecidable").sum())
        a, b = wilson(inc + und, len(s)), wilson(inc, inc + exc)
        rows.append(dict(database=db, sampled=len(s), included=inc, excluded=exc, undecidable=und,
                         title_only=int((s["abstract_source"] == "none").sum()),
                         precision_undecidable_included=a[0], ci_included=f"{a[1]}-{a[2]}",
                         precision_undecidable_left_out=b[0], ci_left_out=f"{b[1]}-{b[2]}"))
    if rows:
        display(pd.DataFrame(rows).set_index("database"))